# DSCI 216 Assignment 2
## Packet Data Analysis, Part I

**Student 1:**  
**Student 2:**  

Complete and submit one notebook per pair. Both students are responsible for understanding the entire submission and being able to explain its reasoning, code, results, and interpretations.

## Purpose and investigation process

In this assignment, you will examine data from Prof. Story's packet-loss experiments and evaluate three AI-generated claims. Read Prof. Story's packet-loss description before beginning.

Your investigation should follow this process:

**claim → investigative questions → evidence needed → probability concepts → computation → results → interpretation and evaluation**

A claim may lead to one or more investigative questions. One well-chosen question may be sufficient; use multiple questions when the claim contains multiple assertions that need separate investigation. 

## Claims to be evaluated

Keep all three claims in mind as you identify and interpret relevant variables during your initial exploration. Each claim is repeated later for easy reference.

> **AI Claim 1:** Packet loss is entirely caused by RcvBufErrors, indicating that the receiver's UDP buffer overflowed. Transfers fail exactly when there are a nonzero number of RcvBufErrors.

> **AI Claim 3:** Unlimited bitrate transfers are the least reliable. When a bitrate limit is imposed, higher bitrates are more reliable. Considering data from both experiments: Unlimited: 86.4% succeeded; 1 Gbit/sec: 99.8% succeeded; 750 Mbit/sec: 96.5% succeeded; 500 Mbit/sec: 90.4% succeeded.

> **AI Claim 4:** The main write method is the least reliable. Considering data from the write method experiment: main: 76.7% succeeded; helper: 99.9% succeeded; defer: 99.9% succeeded; asyncio: 99.9% succeeded.

## Files and setup

Place this notebook and the following files in the same folder:

- `write-method.2026-04-09-1220.csv`
- `max-payload.2026-03-31-1109.csv`

The setup cell below preserves the two datasets separately as `write` and `payload`. It also adds an `experiment` label before combining them as `both`, so the source of each row remains available.

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.options.display.float_format = "{:.4f}".format

write = pd.read_csv("write-method.2026-04-09-1220.csv")
payload = pd.read_csv("max-payload.2026-03-31-1109.csv")

write = write.assign(experiment="write_method")
payload = payload.assign(experiment="max_payload")
both = pd.concat([write, payload], ignore_index=True)

: 

## Coding guidance

Keep your high-level computations clear, organized, and easy to follow. Use meaningful names and display results that directly address your investigative questions. Create and use helper functions when they improve clarity or reduce unnecessary repetition. You may add code and Markdown cells wherever they are useful.

# Part 1: Meet and understand the data

Use Prof. Story's description together with your own inspection of the datasets. Keep this exploration concise and focused on what you need for the three claims. Your interpretations are preliminary: return here and update them if your understanding changes while you investigate the claims.

## 1.1 Inspect the datasets

Examine the dimensions, columns, data types, and a few rows from each dataset. What initial similarities or differences do you notice? Do not draw conclusions about the claims yet.

In [3]:
# Write your inspection code here.
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.options.display.float_format = "{:.4f}".format

write = pd.read_csv("write-method.2026-04-09-1220.csv")
payload = pd.read_csv("max-payload.2026-03-31-1109.csv")

write = write.assign(experiment="write_method")
payload = payload.assign(experiment="max_payload")
both = pd.concat([write, payload], ignore_index=True)

print("--- Dataset: write_method ---")
display(write.head(3))  # Show first few rows
print(f"Dimensions: {write.shape}")
print("Column names and data types:")
display(write.dtypes)
print("\nBasic statistics for numeric columns:")
display(write.describe())

print("\n--- Dataset: max_payload ---")
display(payload.head(3))
print(f"Dimensions: {payload.shape}")
print("Column names and data types:")
display(payload.dtypes)
print("\nBasic statistics for numeric columns:")
display(payload.describe())

--- Dataset: write_method ---


,max_bitrate,regulator_max_bitrate,fec,redundancy,chunk_duration,chunk_max_packets,repair,write_method,max_payload,rcvbuf_size,tx_bytes,tx_packets,OutDatagrams,SndbufErrors,rx_bytes,rx_packets,InDatagrams,InErrors,NoPorts,RcvbufErrors,InCsumErrors,IgnoredMulti,MemErrors,details_uuid,success,returncode,duration,experiment
0,500000000,0,NaN,NaN,NaN,NaN,NaN,defer,65507,NaN,127934818,85869,1909,0,127934818,85869,1909,0,0,0,0,0,0,af0ee69203fa,1,0,2.6179,write_method
1,500000000,0,NaN,NaN,NaN,NaN,NaN,helper,65507,NaN,127934818,85869,1909,0,127934818,85869,1909,0,0,0,0,0,0,1b80b2478dbf,1,0,2.5257,write_method
2,500000000,0,NaN,NaN,NaN,NaN,NaN,asyncio,65507,NaN,127934818,85869,1909,0,127934818,85869,1909,0,0,0,0,0,0,e757d154cac6,1,0,2.6974,write_method


Dimensions: (160000, 28)
Column names and data types:


max_bitrate                int64
regulator_max_bitrate      int64
fec                      float64
redundancy               float64
chunk_duration           float64
chunk_max_packets        float64
repair                   float64
write_method                 str
max_payload                int64
rcvbuf_size              float64
tx_bytes                   int64
tx_packets                 int64
OutDatagrams               int64
SndbufErrors               int64
rx_bytes                   int64
rx_packets                 int64
InDatagrams                int64
InErrors                   int64
NoPorts                    int64
RcvbufErrors               int64
InCsumErrors               int64
IgnoredMulti               int64
MemErrors                  int64
details_uuid                 str
success                    int64
returncode                 int64
duration                 float64
experiment                   str
dtype: object


Basic statistics for numeric columns:


...

,max_bitrate,regulator_max_bitrate,fec,redundancy,chunk_duration,chunk_max_packets,repair,write_method,max_payload,rcvbuf_size,tx_bytes,tx_packets,OutDatagrams,SndbufErrors,rx_bytes,rx_packets,InDatagrams,InErrors,NoPorts,RcvbufErrors,InCsumErrors,IgnoredMulti,MemErrors,details_uuid,success,returncode,duration,experiment
0,500000000,0,NaN,NaN,NaN,NaN,NaN,helper,1472,NaN,128566598,84919,84919,0,128566598,84919,84919,0,0,0,0,0,0,0102eee5568d,1,0,7.5477,max_payload
1,500000000,0,NaN,NaN,NaN,NaN,NaN,helper,9216,NaN,128336608,94944,13564,0,128336608,94944,13564,0,0,0,0,0,0,42344cc5f8ed,1,0,3.0887,max_payload
2,500000000,0,NaN,NaN,NaN,NaN,NaN,helper,65507,NaN,127934818,85869,1909,0,127934818,85869,1909,0,0,0,0,0,0,b3d3b1c3c07a,1,0,2.5167,max_payload


Dimensions: (120000, 28)
Column names and data types:


max_bitrate                int64
regulator_max_bitrate      int64
fec                      float64
redundancy               float64
chunk_duration           float64
chunk_max_packets        float64
repair                   float64
write_method                 str
max_payload                int64
rcvbuf_size              float64
tx_bytes                   int64
tx_packets                 int64
OutDatagrams               int64
SndbufErrors               int64
rx_bytes                   int64
rx_packets                 int64
InDatagrams                int64
InErrors                   int64
NoPorts                    int64
RcvbufErrors               int64
InCsumErrors               int64
IgnoredMulti               int64
MemErrors                  int64
details_uuid                 str
success                    int64
returncode                 int64
duration                 float64
experiment                   str
dtype: object


Basic statistics for numeric columns:


,max_bitrate,regulator_max_bitrate,fec,redundancy,chunk_duration,chunk_max_packets,repair,max_payload,rcvbuf_size,tx_bytes,tx_packets,OutDatagrams,SndbufErrors,rx_bytes,rx_packets,InDatagrams,InErrors,NoPorts,RcvbufErrors,InCsumErrors,IgnoredMulti,MemErrors,success,returncode,duration
count,120000.0000,120000.0000,0.0000,0.0000,0.0000,0.0000,0.0000,120000.0000,0.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000,120000.0000
mean,562500000.0000,0.0000,NaN,NaN,NaN,NaN,NaN,25398.3333,NaN,128279341.4436,88577.3349,33464.0000,0.0000,128279338.6074,88577.3314,33452.9032,11.0966,0.0000,11.0966,0.0000,0.0000,0.0000,0.9213,0.0000,3.0769
std,369756527.0991,0.0000,NaN,NaN,NaN,NaN,NaN,28536.8933,NaN,261083.5952,4518.6068,36694.1354,0.0000,261085.7228,4518.6061,36678.6077,48.3898,0.0000,48.3898,0.0000,0.0000,0.0000,0.2692,0.0000,2.0358
min,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,1472.0000,NaN,127934818.0000,84919.0000,1909.0000,0.0000,127821924.0000,84919.0000,1903.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.3687
25%,375000000.0000,0.0000,NaN,NaN,NaN,NaN,NaN,1472.0000,NaN,127934818.0000,84919.0000,1909.0000,0.0000,127934818.0000,84919.0000,1909.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.4806
50%,625000000.0000,0.0000,NaN,NaN,NaN,NaN,NaN,9216.0000,NaN,128336608.0000,85869.0000,13564.0000,0.0000,128336608.0000,85869.0000,13564.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,2.2718
75%,812500000.0000,0.0000,NaN,NaN,NaN,NaN,NaN,65507.0000,NaN,128566598.0000,94944.0000,84919.0000,0.0000,128566598.0000,94944.0000,84919.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,3.6776
max,1000000000.0000,0.0000,NaN,NaN,NaN,NaN,NaN,65507.0000,NaN,128566668.0000,94945.0000,84919.0000,0.0000,128566598.0000,94944.0000,84919.0000,1781.0000,0.0000,1781.0000,0.0000,0.0000,0.0000,1.0000,0.0000,7.5600


**Response:**  
The first difference that caught my eye with the two csv files is that the dimension for write_method.csv is signficantly larger than max-payload.csv, with write_method.csv containing 20,000 more dimensions than max-payload.csv. The columns and data types across each csv seem to be the same either being integers or floats. However, columns fed, redundancy, chunk_duration, chunk_max_packets, repair, and rcvbuf_size all seem to be NaN possibly being values written in hexadecimal instead of integers or floats. write_method.csv also had 4 more columns than max-payload.csv containing an extra defer, helper, asyncio, and main columns.

## 1.2 Interpret the observational unit and relevant variables

Based on the experiment description, the claims, and your inspection:

- Which variables appear relevant to the three claims? What does each represent, what role does it play, and what values can it take?
- Do any values require special interpretation?
- What does one row represent? Describe it in the context of the experiment.

You do not need to describe every column. Update or refine this response later if you learn more during the claim investigations.

In [ ]:
# Inspect the variables and values needed to support your response.
Variables: RcvBufErrors, success, rx_packets, max_bitrate, regulator_max_bitrate

The variables that matter here split cleanly by claim. For Claim 1, RcvbufErrors (packets dropped for a full receive buffer) and InErrors (a kernel UDP error counter) turn out to be the exact same value in every row, so they're one variable, not two independent checks. tx_packets/rx_packets give a second, packet-count-based view of "loss," and success is the binary pass/fail outcome the claim is actually about. For Claim 3, the relevant variable is max_bitrate (the sending-rate cap, 0/500M/750M/1G, with 0 meaning "unlimited") against success, using both experiments combined. For Claim 4 it's write_method (main/helper/defer/asyncio) against success, but only within the write_method experiment, since max_payload rows hold write_method fixed at helper.

A few values need care before trusting them at face value. Nonzero RcvbufErrors lines up with success == 0 almost perfectly, but not exactly — 5 of 280,000 rows fail with RcvbufErrors == 0, which already complicates Claim 1's "exactly when" wording. Also, raw tx/rx packet-count mismatches are essentially absent even where RcvbufErrors is nonzero, so RcvbufErrors isn't simply counting undelivered packets — it's a different signal than "packet loss" in the literal sense. And since Claim 3 mixes both experiments, it's implicitly averaging over different fixed write_method/max_payload settings, which is worth keeping in mind when interpreting differences by bitrate.

A single row represents one trial run of the harness: one UDP transfer attempt under a specific parameter configuration, with the resulting socket counters and a pass/fail outcome recorded. details_uuid is unique across all 280,000 rows, confirming each row is an independent run rather than a repeated measurement.

In [4]:
# --- Variables relevant to Claim 1 (RcvBufErrors -> packet loss -> failure) ---
print(both[["RcvbufErrors", "InErrors"]].describe())
print("InErrors == RcvbufErrors in every row?", (both["InErrors"] == both["RcvbufErrors"]).all())

# packet-count loss proxy
loss = both["tx_packets"] - both["rx_packets"]
print(loss.value_counts().head())

# does success align perfectly with RcvbufErrors == 0?
print(pd.crosstab(both["success"], both["RcvbufErrors"] > 0))

# the exceptions to a perfect alignment
edge_cases = both[(both["success"] == 0) & (both["RcvbufErrors"] == 0)]
print(edge_cases[["experiment", "write_method", "max_bitrate", "max_payload",
                   "RcvbufErrors", "tx_packets", "rx_packets", "duration", "success"]])

# --- Variables relevant to Claim 3 (bitrate vs. reliability, both experiments) ---
print(both.groupby("max_bitrate")["success"].agg(["mean", "count"]))

# --- Variables relevant to Claim 4 (write_method vs. reliability, write_method experiment only) ---
print(write["write_method"].value_counts())
print(write.groupby("write_method")["success"].mean())

# --- What one row represents ---
print(both["details_uuid"].duplicated().sum(), "duplicate uuids")
display(both.sample(1, random_state=0))

       RcvbufErrors    InErrors
count   280000.0000 280000.0000
mean         5.3694      5.3694
std         32.2425     32.2425
min          0.0000      0.0000
25%          0.0000      0.0000
50%          0.0000      0.0000
75%          0.0000      0.0000
max       1781.0000   1781.0000
InErrors == RcvbufErrors in every row? True
0     279616
1        378
19         1
12         1
26         1
Name: count, dtype: int64
RcvbufErrors   False  True 
success                    
0                  5  18820
1             261175      0
          experiment write_method  max_bitrate  max_payload  RcvbufErrors  \
143323  write_method         main   1000000000        65507             0   
149769  write_method       helper   1000000000        65507             0   
152268  write_method        defer            0        65507             0   
239607   max_payload       helper            0        65507             0   
248906   max_payload       helper            0         9216             0   

  

,max_bitrate,regulator_max_bitrate,fec,redundancy,chunk_duration,chunk_max_packets,repair,write_method,max_payload,rcvbuf_size,tx_bytes,tx_packets,OutDatagrams,SndbufErrors,rx_bytes,rx_packets,InDatagrams,InErrors,NoPorts,RcvbufErrors,InCsumErrors,IgnoredMulti,MemErrors,details_uuid,success,returncode,duration,experiment
143427,500000000,0,NaN,NaN,NaN,NaN,NaN,main,65507,NaN,127934818,85869,1909,0,127934818,85869,1893,16,0,16,0,0,0,ee4f001449d9,0,0,2.5160,write_method


**Response:**  
Write your response here.

## 1.3 Understand the organization and readiness of the data

Use concise summaries to determine how trials are distributed across experiments and configurations. Check the key variables for missing, unusual, or encoded values. What features of the experimental design or data organization should you keep in mind when combining or comparing observations?

In [4]:
# Write the concise checks needed to support your response.

**Response:**  
Write your response here.

# Part 2: Claim 1 — RcvBufErrors and transfer failures

> **AI Claim 1:** Packet loss is entirely caused by RcvBufErrors, indicating that the receiver's UDP buffer overflowed. Transfers fail exactly when there are a nonzero number of RcvBufErrors.

The CSV column is named `RcvbufErrors`.

## 2.1 Investigative question

For this first claim, an investigative question is provided as a model:

> **What relationship do the observed data show between `RcvbufErrors` and transfer failure, and does that relationship support the claim that `RcvbufErrors` fully explains packet loss?**

Briefly explain how this question captures the important parts of the claim.

**Response:**  
Write your response here.

## 2.2 Evidence and probability reasoning

1. What evidence would you need to answer the investigative question and evaluate the claim?
2. What probability concepts are related to this evidence? Explain how they are related.

**Response:**  
Write your response here.

## 2.3 Obtain the evidence

Use the data to obtain the evidence you identified. Your code and displayed results should make it clear how the evidence answers the investigative question.

In [5]:
# Write your Claim 1 analysis here. Add cells if needed.

## 2.4 Results, interpretation, and evaluation

- What did you find? Report the results needed to answer your investigative question.
- What conclusion is supported by your evidence?
- Evaluate the AI claim: Are any stated calculations correct? Is the evidence appropriate and sufficient? Is the conclusion justified?
- If the claim is not fully justified, revise it to state a conclusion that the evidence supports.

**Response:**  
Write your response here.

# Part 3: Claim 3 — bitrate and reliability

> **AI Claim 3:** Unlimited bitrate transfers are the least reliable. When a bitrate limit is imposed, higher bitrates are more reliable. Considering data from both experiments: Unlimited: 86.4% succeeded; 1 Gbit/sec: 99.8% succeeded; 750 Mbit/sec: 96.5% succeeded; 500 Mbit/sec: 90.4% succeeded.

## 3.1 Formulate the investigative question(s)

This claim contains reported numerical results and broader conclusions about bitrate and reliability. Write one or more investigative questions that would allow you to evaluate the important parts of the claim using the available data. Briefly explain how your question or questions connect to the claim.

**Investigative question(s) and explanation:**  
Write your response here.

## 3.2 Evidence and probability reasoning

1. What evidence would you need to answer your investigative question(s) and evaluate the claim?
2. What probability concepts are related to this evidence? Explain how they are related.

**Response:**  
Write your response here.

## 3.3 Obtain the evidence

Use the data to obtain the evidence you identified. Your code and displayed results should make it clear how the evidence answers your investigative question(s).

In [6]:
# Write your Claim 3 analysis here. Add cells if needed.

## 3.4 Results, interpretation, and evaluation

- What did you find? Report the results needed to answer your investigative question(s).
- What conclusion is supported by your evidence?
- Evaluate the AI claim: Are its stated calculations correct? Is its evidence appropriate and sufficient? Is its conclusion justified?
- If the claim is not fully justified, revise it to state a conclusion that the evidence supports.

**Response:**  
Write your response here.

# Part 4: Claim 4 — write method and reliability

> **AI Claim 4:** The main write method is the least reliable. Considering data from the write method experiment: main: 76.7% succeeded; helper: 99.9% succeeded; defer: 99.9% succeeded; asyncio: 99.9% succeeded.

## 4.1 Formulate the investigative question(s)

Translate the claim into one or more investigative questions that can be answered using the available data. Briefly explain how your question or questions capture the important parts of the claim.

**Investigative question(s) and explanation:**  
Write your response here.

## 4.2 Evidence and probability reasoning

1. What evidence would you need to answer your investigative question(s) and evaluate the claim?
2. What probability concepts are related to this evidence? Explain how they are related rather than only listing terms.

**Response:**  
Write your response here.

## 4.3 Obtain the evidence

Use the data to obtain the evidence you identified. Your code and displayed results should make it clear how the evidence answers your investigative question(s).

In [7]:
# Write your Claim 4 analysis here. Add cells if needed.

## 4.4 Results, interpretation, and evaluation

- What did you find? Report the results needed to answer your investigative question(s).
- What conclusion is supported by your evidence?
- Evaluate the AI claim: Are its stated calculations correct? Is its evidence appropriate and sufficient? Is its conclusion justified?
- If the claim is not fully justified, revise it to state a conclusion that the evidence supports.

**Response:**  
Write your response here.

# Part 5: Overall synthesis

Using examples from your investigations, explain the difference between:

- a numerically correct calculation,
- appropriate and sufficient evidence, and
- a justified conclusion.

**Response:**  
Write your response here.

# Part 6: Collaboration and AI-use record

Complete this section jointly as a pair.

1. Briefly describe how you organized your collaboration.
2. Identify any AI tools used and explain what they were used for. If the partners used AI differently, describe each person's use separately. If you did not use AI, state that.
3. If AI was used, how did you review, test, or verify any AI-assisted work?

**Response:**  
Write your response here.

# Part 7: Individual reflections

Each student must write a separate response. In a paragraph, describe one important thing you learned from this assignment about asking questions, selecting appropriate evidence, using probability concepts, or interpreting computational results. 

## Student 1 reflection

**Name:**  

Write your individual reflection here.

## Student 2 reflection

**Name:**  

Write your individual reflection here.

# Before submitting

- Confirm that both students' names appear at the top.
- Restart the kernel and run all cells from beginning to end.
- Submit one completed `.ipynb` file for the pair.